In [1]:
import pandas as pd

Data integration and join valdiation

combine the cleaned datasets safely, validate table relationships, and prepare analysis ready datasets without losing or multiplying records

In [2]:
customers = pd.read_csv("../06_clean_data/customers_clean.csv")
orders = pd.read_csv("../06_clean_data/orders_clean.csv")
order_items = pd.read_csv("../06_clean_data/order_items_clean.csv")
products = pd.read_csv("../06_clean_data/products_clean.csv")
payments = pd.read_csv("../06_clean_data/payments_clean.csv")
reviews = pd.read_csv("../06_clean_data/reviews_clean.csv")
sellers = pd.read_csv("../06_clean_data/sellers_clean.csv")
geolocation = pd.read_csv("../06_clean_data/geolocation_clean.csv")
category_translation = pd.read_csv(
    "../06_clean_data/category_translation_clean.csv"
)

In [3]:
print("cleaned datasets loaded successfully")

cleaned datasets loaded successfully


In [4]:
missing_customers = (
    ~orders["customer_id"].isin(customers["customer_id"])
).sum()
print("orders with no matching customer",missing_customers)

orders with no matching customer 0


In [5]:
missing_order_items = (
    ~orders["order_id"].isin(order_items["order_id"])
).sum()
print("orders with no matching order items",missing_order_items)

orders with no matching order items 775


there are 775 orders that exist in the orders table, but those same order_id values re not present in the order items table

In [6]:
missing_products = (
    ~order_items["product_id"].isin(products["product_id"])
).sum()

print("Order item rows with no matching product:", missing_products)

Order item rows with no matching product: 0


In [7]:
missing_sellers = (
    ~order_items["seller_id"].isin(sellers["seller_id"])
).sum()

print("order item rows with no matching seller",missing_sellers)

order item rows with no matching seller 0


In [8]:
missing_payments = (
    ~orders["order_id"].isin(payments["order_id"])
).sum()
print("order with no matching payment",missing_payments)

order with no matching payment 1


In [9]:
missing_reviews = (
    ~orders["order_id"].isin(reviews["order_id"])
).sum()
print("orders with no matching reviews",missing_reviews)

orders with no matching reviews 768


In [10]:
missing_customer_geo = (
    ~customers["customer_zip_code_prefix"]
    .isin(geolocation["geolocation_zip_code_prefix"])
).sum()
print("customer with no matching geolocation",missing_customer_geo)

customer with no matching geolocation 278


278 customer records have a postcode that does not exist in the cleaned geolocation table

In [11]:
missing_seller_geo = (
    ~sellers["seller_zip_code_prefix"]
    .isin(geolocation["geolocation_zip_code_prefix"])
).sum()
print("sellers with no matching geolocation",missing_seller_geo)

sellers with no matching geolocation 7


7 seller records have postcode that does not exist in the cleaned geolocation table

### Joining the Tables

In [12]:
orders_customers = orders.merge(
    customers,
    on = "customer_id",
    how = "left"
)

In [13]:
print("Orders before join:", orders.shape[0])
print("Rows after join:", orders_customers.shape[0])

Orders before join: 99441
Rows after join: 99441


In [14]:
order_items = pd.read_csv(
    "../06_clean_data/order_items_clean.csv"
)

In [15]:
orders_items = orders_customers.merge(
    order_items,
    on="order_id",
    how="left"
)

In [16]:
print("Orders before join:", orders_customers.shape[0])
print("Rows after Order Items join:", orders_items.shape[0])

Orders before join: 99441
Rows after Order Items join: 113425


The order item has one to many relationship 

In [17]:
orders_items_products = orders_items.merge(
    products,
    on = "product_id",
    how = "left"
)

In [18]:
print("Rows before Products join:", orders_items.shape[0])
print("Rows after Products join:", orders_items_products.shape[0])

Rows before Products join: 113425
Rows after Products join: 113425


In [19]:
orders_items_products = orders_items_products.rename(
    columns={
        "missing_geolocation": "customer_missing_geolocation"
    }
)

sellers_for_join = sellers.rename(
    columns={
        "missing_geolocation": "seller_missing_geolocation"
    }
)

In [20]:
orders_items_products_sellers = orders_items_products.merge(
    sellers_for_join,
    on = "seller_id",
    how = "left"
)

In [21]:
print("Rows before Sellers join:", orders_items_products.shape[0])
print("Rows after Sellers join:", orders_items_products_sellers.shape[0])

Rows before Sellers join: 113425
Rows after Sellers join: 113425


changing the geolocation name 

In [22]:
customer_geo = geolocation.rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix",
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng",
    "geolocation_city": "customer_geo_city",
    "geolocation_state": "customer_geo_state"
})

In [23]:
orders_full = orders_items_products_sellers.merge(
    customer_geo,
    on = "customer_zip_code_prefix",
    how = "left"
)

In [24]:
print("Rows before Customer Geolocation join:",
      orders_items_products_sellers.shape[0])

print("Rows after Customer Geolocation join:",
      orders_full.shape[0])

Rows before Customer Geolocation join: 113425
Rows after Customer Geolocation join: 113425


In [25]:
payment_summary = (
    payments
    .groupby("order_id")
    .agg(
        total_payment_value = ("payment_value","sum"),
        payment_record_count = ("payment_sequential","count"),
        max_installments = ("payment_installments","max")
    )
    .reset_index()
)

In [26]:
print("Payment rows before summary:", payments.shape[0])
print("Payment summary rows:", payment_summary.shape[0])
print("Duplicate order IDs:", payment_summary["order_id"].duplicated().sum())

Payment rows before summary: 103886
Payment summary rows: 99440
Duplicate order IDs: 0


In [27]:
seller_geo = geolocation.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix",
    "geolocation_lat": "seller_lat",
    "geolocation_lng": "seller_lng",
    "geolocation_city": "seller_geo_city",
    "geolocation_state": "seller_geo_state"
})

In [28]:
orders_full_geo = orders_full.merge(
    seller_geo,
    on="seller_zip_code_prefix",
    how="left"
)

In [29]:
print("Rows before Seller Geolocation join:", orders_full.shape[0])
print("Rows after Seller Geolocation join:", orders_full_geo.shape[0])

Rows before Seller Geolocation join: 113425
Rows after Seller Geolocation join: 113425


In [30]:
orders_with_payments = orders_full_geo.merge(
    payment_summary,
    on="order_id",
    how="left"
)

In [31]:
print("Rows before Payments join:", orders_full_geo.shape[0])
print("Rows after Payments join:", orders_with_payments.shape[0])

Rows before Payments join: 113425
Rows after Payments join: 113425


In [32]:
review_summary = (
    reviews
    .groupby("order_id")
    .agg(
        review_count = ("review_id","count"),
        average_review_count = ("review_score","mean")
    )
    .reset_index()
)

In [33]:
print("Review rows before summary:", reviews.shape[0])
print("Review summary rows:", review_summary.shape[0])
print(
    "Duplicate order IDs:",
    review_summary["order_id"].duplicated().sum()
)

Review rows before summary: 99224
Review summary rows: 98673
Duplicate order IDs: 0


In [34]:
orders_integrated = orders_with_payments.merge(
    review_summary,
    on = "order_id",
    how = "left"
)

In [35]:
print("Rows before Reviews join:", orders_with_payments.shape[0])
print("Rows after Reviews join:", orders_integrated.shape[0])

Rows before Reviews join: 113425
Rows after Reviews join: 113425


Checking for the final structure

In [36]:
print("Total integrated rows:", orders_integrated.shape[0])
print("Unique orders:", orders_integrated["order_id"].nunique())
print("Unique customers:", orders_integrated["customer_id"].nunique())
print("Unique products:", orders_integrated["product_id"].nunique())
print("Unique sellers:", orders_integrated["seller_id"].nunique())

Total integrated rows: 113425
Unique orders: 99441
Unique customers: 99441
Unique products: 32951
Unique sellers: 3095


Creating the order, payment, review summary orderlevel table

In [37]:
order_summary = orders_customers.merge(
    customer_geo,
    on = "customer_zip_code_prefix",
    how = "left"
)

In [38]:
order_summary = order_summary.merge(
    payment_summary,
    on="order_id",
    how="left"
)

In [39]:
order_summary = order_summary.merge(
    review_summary,
    on="order_id",
    how="left"
)

In [40]:
print("Order summary rows:", order_summary.shape[0])
print("Unique order IDs:", order_summary["order_id"].nunique())
print("Duplicate order IDs:", order_summary["order_id"].duplicated().sum())

Order summary rows: 99441
Unique order IDs: 99441
Duplicate order IDs: 0


Final integrated datasets for analysis

In [41]:
import os

os.makedirs("../07_integrated_data", exist_ok=True)

orders_integrated.to_csv(
    "../07_integrated_data/orders_integrated.csv",
    index=False
)

order_summary.to_csv(
    "../07_integrated_data/order_summary.csv",
    index=False
)

print("Integrated tables saved successfully.")

Integrated tables saved successfully.


In [42]:
import os

os.listdir("../07_integrated_data")

['.ipynb_checkpoints',
 'orders_integrated.csv',
 'order_summary.csv',
 'Untitled.ipynb']